In [1]:
import gym
import numpy as np
import random
from IPython.display import clear_output

In [2]:
training_env = gym.make("Taxi-v3")

In [3]:
action_space_size = training_env.action_space.n
state_space_size = training_env.observation_space.n

q_table = np.zeros((state_space_size, action_space_size))

In [4]:
#SET 1
episodes_nr = 10_000
maximum_steps = 1000
learning_rate = 0.1
discount_rate = 0.99
exploration_rate = 1
minimum_exploration_rate = 0.01
maximum_exploration_rate = 1
exploration_rate_decay_rate = 0.001

In [5]:
'''
#SET 2
episodes_nr = 11_600
last_step_ponder = 0.001
discount_rate = 0.99
maximum_steps = int(np.log(last_step_ponder) / np.log(discount_rate)) #optimal
learning_rate = 1_000 / episodes_nr # optimal

exploration_rate = 1
minimum_exploration_rate = 0.01
maximum_exploration_rate = 1

exploration_rate_decay_rate = -np.log(0.00001)/episodes_nr #optimal
'''

'\n#SET 2\nepisodes_nr = 11_600\nlast_step_ponder = 0.001\ndiscount_rate = 0.99\nmaximum_steps = int(np.log(last_step_ponder) / np.log(discount_rate)) #optimal\nlearning_rate = 1_000 / episodes_nr # optimal\n\nexploration_rate = 1\nminimum_exploration_rate = 0.01\nmaximum_exploration_rate = 1\n\nexploration_rate_decay_rate = -np.log(0.00001)/episodes_nr #optimal\n'

In [6]:
rewards_all_episodes = []
for episode in range(episodes_nr):
    state,info = training_env.reset()
    done = False
    rewards_current_episode = 0
    mask = info["action_mask"]
    for step in range(maximum_steps):
        exploration_rate_threshhold = random.uniform(0,1)
        if exploration_rate_threshhold < exploration_rate:
            #explore
            valid_actions = np.where(mask)[0]
            action = np.random.choice(valid_actions)
            #action = training_env.action_space.sample()
        else:
            #exploit
            action = np.argmax(q_table[state,:])

        new_state, reward, terminated, truncated, info = training_env.step(action)

        mask = info["action_mask"]
        done = terminated or truncated
        
        q_table[state, action] = (1 - learning_rate) * q_table[state,action] + learning_rate * (reward + discount_rate * np.max(q_table[new_state, :]))
        state = new_state
        rewards_current_episode += reward

        if done:
            break

    rewards_all_episodes.append(rewards_current_episode)
    exploration_rate = minimum_exploration_rate + (maximum_exploration_rate - minimum_exploration_rate) * np.exp(-exploration_rate_decay_rate * episode)

rewards_per_thousands_episodes = np.array_split(np.array(rewards_all_episodes), episodes_nr // 1000)

print("Average reward per thousand episodes:")
count = 1000
for r in rewards_per_thousands_episodes:
    avg_reward = np.mean(r)
    print(f"{count}: {avg_reward:.2f}")
    count += 1000


Average reward per thousand episodes:
1000: -148.49
2000: -4.70
3000: 5.91
4000: 7.28
5000: 7.70
6000: 7.78
7000: 7.55
8000: 7.81
9000: 7.58
10000: 7.60


In [7]:
for episode in range(5):
    env = gym.make("Taxi-v3", render_mode="human")
    state,_ = env.reset()
    for step in range(maximum_steps):
        env.render()
        
        action = np.argmax(q_table[state,:])
        new_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        if done:
            env.render()            
            break

        state = new_state
    env.close()